# Problem 1 — Exploratory diagnostics

Companion notebook for analyses that are **not** required to run the main first-pass **index vs farmer** classifier, but help interpret the panel: **missingness**, **price-level visuals**, **PCA on row-demeaned residuals**, **residual variance**, **residual-noise diagnostics** (bias, pooled shape, skew/kurtosis), **spectral flatness**, **lag-1 autocorrelation**, and **NNLS** leave-one-out \(R^2\) on residuals / levels / first differences.

Run the streamlined pipeline in **`Problem1.ipynb`** for `index_feature_table` and submission helpers.


## Contents

1. **Workspace** — load the panel and shared helpers.
2. **Missing data** — total NaNs, sparsity by column, contiguous vs random gaps.
3. **Price levels** — cross-sectional means, aggregate trend, all series overlay.
4. **PCA on row-demeaned residuals** — variance explained, loadings, participation mass.
5. **Residual variance** — per-column variance of cross-sectional residuals.
6. **Residual noise** — bias, pooled distribution, skew/kurtosis, spectral flatness, lag-1 ACF.
7. **NNLS explainability** — leave-one-out \(R^2\) on residuals, levels, and first differences.


In [ ]:
import platform
import typing as t

import numpy as np
import pandas as pd
import scipy
import statsmodels.api as sm
import xgboost as xgb
import torch

import matplotlib
import matplotlib.pyplot as plt
import plotly

from scipy.optimize import nnls
from scipy.signal import periodogram

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer


### Workspace


In [ ]:
DF = pd.read_csv("limestone_data_challenge_2026.data.csv")
DF = DF.set_index(keys="time")

DF.head()

In [ ]:
# Shared helpers used across later sections

def nnls_leave_one_out_r2(Z: np.ndarray, col_names) -> pd.Series:
    """Each column j: predict y = Z[:,j] from all other columns with non-negative coefficients."""
    _, n = Z.shape
    out = {}
    for j in range(n):
        y = Z[:, j].astype(np.float64, copy=False)
        X = np.delete(Z, j, axis=1)
        ss_tot = np.sum((y - y.mean()) ** 2)
        if ss_tot < 1e-18:
            out[col_names[j]] = np.nan
            continue
        beta, _ = nnls(X, y)
        pred = X @ beta
        ss_res = np.sum((y - pred) ** 2)
        out[col_names[j]] = float(1.0 - ss_res / ss_tot)
    return pd.Series(out, name="r2_nnls")


def periodogram_flatness(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size < 64:
        return np.nan
    _, pxx = periodogram(x, detrend="constant")
    pxx = np.maximum(pxx, 1e-20)
    return float(np.exp(np.mean(np.log(pxx))) / np.mean(pxx))


def lag1_acf_calendar(series: pd.Series) -> float:
    """Pearson correlation between r_t and r_{t-1} using rows where both exist."""
    prev = series.shift(1)
    m = series.notna() & prev.notna()
    if m.sum() < 5:
        return np.nan
    x = series[m].to_numpy(dtype=float)
    y = prev[m].to_numpy(dtype=float)
    return float(np.corrcoef(x, y)[0, 1])


## Understanding the distribution of Nans:

It would be prudcent to understand if only prices hae NaN values or indexes have NaN values too, we should look at the ditribution of NaNs amongst columns to first of all understand the data

In [ ]:
# total NaNs in the data
total_nans = DF.isna().sum().sum()
print(f"Total NaN ={total_nans}")

#30-60% NaNs

#Nans per column of data 
columns_all_defined = DF.isna().sum().eq(0).sum()
print(columns_all_defined)

So quite clearly all columns have NaN values including both prices and index data even though the question didn't explictly state this, there was ambiguity about what smudged "prices" was referring to be it is quite claer now that his was referring to both index prices and the actual prices of the columns.

### Understanding how NaNs are distributed in columns 

In [ ]:
percent_undefined = DF.isna().sum()/len(DF.index)
percent_undefined.hist()

Quite clearly the data is extremely sparse with many columns having a range oof 0.30-0.60 of their data missing 

### Is data missing in contiguous blocks within columns or is it randomly missing? 

This has secondary impacts for inference when we build our trading strategies and may also inform some other aspects of our analysis. 

In [ ]:

# Row order is treated as the time axis: contiguous NaNs = multi-day gaps in that series.
is_na = DF.isna()
n_rows, n_cols = is_na.shape

print("=== Whole dataset ===")
print(f"Share of (row, column) cells missing: {is_na.to_numpy().mean():.4f}")

na_per_row = is_na.sum(axis=1)
print(
    "NAs per row (how many series missing on the same day): "
    f"min={na_per_row.min()}, median={na_per_row.median():.0f}, max={na_per_row.max()}"
)

na_per_row.plot.hist(bins=30, figsize=(8, 4), edgecolor="white")
plt.xlabel("Number of columns NA on that row")
plt.ylabel("Count of rows")
plt.title("Co-missingness across columns on the same day")
plt.tight_layout()
plt.show()

# Do gaps tend to occur on the same calendar rows across columns?
miss_corr = is_na.astype(float).corr()
triu = np.triu_indices_from(miss_corr, k=1)
off_diag = miss_corr.to_numpy()[triu]
print(
    "\nPairwise correlation of missingness indicators (off-diagonal): "
    f"median={np.median(off_diag):.3f}, mean={off_diag.mean():.3f}"
)
print("(Higher values → missingness more aligned in time across series.)")

plt.figure(figsize=(7, 5.5))
plt.imshow(miss_corr, aspect="auto", vmin=-1, vmax=1, cmap="coolwarm")
plt.colorbar(label="corr(indicator missing)")
plt.xlabel("Column (index in DF)")
plt.ylabel("Column (index in DF)")
plt.title("Missingness co-movement across columns")
plt.tight_layout()
plt.show()


def na_run_lengths(s: pd.Series) -> np.ndarray:
    """Lengths of each contiguous block of NaNs along the time axis."""
    mask = s.isna()
    if not mask.any():
        return np.array([], dtype=int)
    grp = (mask != mask.shift(fill_value=False)).cumsum()
    lengths = []
    for _, block in mask.groupby(grp):
        if bool(block.iloc[0]):
            lengths.append(int(block.sum()))
    return np.asarray(lengths, dtype=int)


rows = []
for col in DF.columns:
    lens = na_run_lengths(DF[col])
    n_na = int(DF[col].isna().sum())
    p = n_na / n_rows
    exp_runs = p + (n_rows - 1) * p * (1 - p) if n_rows else 0.0
    n_runs = len(lens)
    if n_runs == 0:
        rows.append(
            {
                "column": col,
                "n_na": n_na,
                "n_runs": 0,
                "mean_run": np.nan,
                "median_run": np.nan,
                "max_run": np.nan,
                "runs_vs_iid": np.nan,
            }
        )
    else:
        rows.append(
            {
                "column": col,
                "n_na": n_na,
                "n_runs": n_runs,
                "mean_run": float(lens.mean()),
                "median_run": float(np.median(lens)),
                "max_run": int(lens.max()),
                "runs_vs_iid": n_runs / exp_runs if exp_runs > 0 else np.nan,
            }
        )

run_summary = pd.DataFrame(rows).set_index("column").sort_values("mean_run", ascending=False)

print("\n=== Per-column contiguous NaN runs (time-ordered rows) ===")
print(
    "runs_vs_iid = (# observed NA runs) / (expected # runs if each row were NA "
    "independently with that column's marginal rate p).\n"
    "  < 1  → longer blocks / fewer runs than iid (gaps clump in time)\n"
    "  ~ 1  → similar fragmentation to iid at random\n"
    "  > 1  → shorter runs / more alternation than iid\n"
)
print("Largest mean contiguous NA length (rows ≈ days):")
print(run_summary.head(10).to_string())
print("\nSmallest mean contiguous NA length:")
print(run_summary.tail(10).to_string())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
run_summary["mean_run"].dropna().hist(ax=axes[0], bins=20, edgecolor="white")
axes[0].set_xlabel("Mean length of a NA run (rows)")
axes[0].set_ylabel("# columns")
axes[0].set_title("Across columns: typical gap length")

run_summary["runs_vs_iid"].dropna().hist(ax=axes[1], bins=20, edgecolor="white")
axes[1].axvline(1.0, color="k", linestyle="--", linewidth=1)
axes[1].set_xlabel("Observed runs / iid-expected runs")
axes[1].set_ylabel("# columns")
axes[1].set_title("Clustering vs iid benchmark")
plt.tight_layout()
plt.show()


## Understanding the distribution of prices

For each of the columns it would be great to see how different prices may evolve through time to see if there are any outliers when considering indexes vs farmers, and to potentially see how correlated prices from farmers are. Do they in general move together, or might there be quite a bit of variance. 


There might be several different views to get a perspective on this so let's plot a variety of different views to understand pricing dynamics in the market through time

### Average prices per index and farmer

In [ ]:
price_means = DF.mean()
price_means.hist()
price_means.describe()

### Average price over time 

In [ ]:
prices_over_time = DF.mean(axis=1)
prices_over_time.plot()

The time series of prices looks to be generally an increasing trend of prices, futhermore it looks like it's mean reverting to a general trend. It looks like the times series of the average price would be stationary once differenced with a linear model against time. This may inform our trading strategy, although we need to be careful not to overfit the data.

### Individual prices over this time

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for col in DF.columns:
    ax.plot(DF.index, DF[col], linewidth=0.6, alpha=0.35)

ax.set_xlabel("time")
ax.set_ylabel("price")
ax.set_title("All price series (NaNs create gaps; no interpolation)")
plt.tight_layout()
plt.show()

# PCA analysis of time series data

In [ ]:

# Numeric price columns only (same panel as DF)
price_cols = DF.select_dtypes(include=[np.number]).columns
X_raw = DF[price_cols].copy()

# Row-mean fill: each NA at time t is filled with the mean of *observed* prices that row (cross-section that day)
row_means = X_raw.mean(axis=1)
DF_filled_row_mean = X_raw.T.fillna(row_means).T
# If a whole row is NaN, row mean is NaN — fall back to 0 for PCA
DF_filled_row_mean = DF_filled_row_mean.fillna(0.0)

row_medians = X_raw.median(axis=1)
X_median = X_raw.T.fillna(row_medians).T.fillna(0.0)

# Time-ordered simple fill: carry last known, then next known, then column mean
X_ffill = X_raw.ffill().bfill().fillna(X_raw.mean()).fillna(0.0)

# Row/column structured imputation (slower; uses cross-sectional info)
_iter = IterativeImputer(max_iter=15, random_state=0)
X_iter = pd.DataFrame(
    _iter.fit_transform(X_raw),
    index=X_raw.index,
    columns=X_raw.columns,
)


def pca_report(X: pd.DataFrame, title: str) -> np.ndarray:
    pipe = Pipeline(
        [
            ("scale", StandardScaler()),
            ("pca", PCA()),
        ]
    )
    pipe.fit(X)
    pca = pipe.named_steps["pca"]
    evr = pca.explained_variance_ratio_
    cum = np.cumsum(evr)
    print(f"\n=== {title} ===")
    for thr in (0.80, 0.90, 0.95):
        k = int(np.searchsorted(cum, thr) + 1)
        print(f"  PCs for ≥{thr:.0%} variance: {k} (cumulative {cum[k - 1]:.4f})")
    print(
        f"  PC1–3 share: {evr[0]:.4f}, {evr[1]:.4f}, {evr[2]:.4f} "
        f"(total first 3: {cum[2]:.4f})"
    )
    return cum


datasets = {
    "Mean impute (row avg per time)": DF_filled_row_mean,
    "Median impute (row per time)": X_median,
    "ffill → bfill → mean": X_ffill,
    "IterativeImputer": X_iter,
}

curves = {name: pca_report(data, name) for name, data in datasets.items()}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(1, len(next(iter(curves.values()))) + 1)
for name, cum in curves.items():
    axes[0].plot(x, cum, label=name, linewidth=1.5)
axes[0].axhline(0.9, color="k", linestyle="--", linewidth=0.8, alpha=0.6)
axes[0].axhline(0.95, color="k", linestyle=":", linewidth=0.8, alpha=0.6)
axes[0].set_xlabel("Number of PCs")
axes[0].set_ylabel("Cumulative explained variance")
axes[0].set_title("PCA after imputation (scaled features)")
axes[0].legend(loc="lower right", fontsize=8)
axes[0].set_ylim(0, 1.02)

# Scree for the mean-imputed panel (primary baseline)
pipe_mean = Pipeline(
    [("scale", StandardScaler()), ("pca", PCA())]
)
pipe_mean.fit(DF_filled_row_mean)
evr_m = pipe_mean.named_steps["pca"].explained_variance_ratio_
axes[1].plot(np.arange(1, min(21, len(evr_m) + 1)), evr_m[:20], marker="o", ms=3)
axes[1].set_xlabel("PC")
axes[1].set_ylabel("Explained variance ratio")
axes[1].set_title("Scree (first 20 PCs, row-mean impute)")
plt.tight_layout()
plt.show()


### PCA on row-demeaned residuals

For each day (row), take the **cross-sectional mean** of observed prices, then form **residuals** = actual price minus that mean. This removes the common “level of the market” that day and focuses on **relative** cheapness/ richness across series.

**Missing values:** residuals are undefined where the price is missing. We set those to **0**, which is the same as imputing the missing level with the row mean before subtracting (so no deviation from the cross-sectional average).

PCA is then fit on this residual matrix (scaled) to see what explains **co-movement in relative prices**.

**Next:** PC **loadings** (which columns sit on the leading factors) — subsection immediately after the PCA code cell.


In [ ]:

price_cols = DF.select_dtypes(include=[np.number]).columns
X_raw = DF[price_cols].copy()

# Cross-sectional mean each day (over non-missing prices in that row)
row_means = X_raw.mean(axis=1)
# Residual: column value minus row mean; NaN where price was missing
R = X_raw.sub(row_means, axis=0)
# Missing → 0 residual (equivalent to level imputed by row mean before demeaning)
R = R.fillna(0.0)

print("R_ij = price_ij − mean_t(prices on day t | observed). Missing cells → 0.")
print(f"Shape {R.shape}. Row sums (exact 0 whenever that day has ≥1 price): min={R.sum(axis=1).min():.6f}, max={R.sum(axis=1).max():.6f}")

pipe_res = Pipeline(
    [
        ("scale", StandardScaler()),
        ("pca", PCA()),
    ]
)
pipe_res.fit(R)
pca = pipe_res.named_steps["pca"]
evr = pca.explained_variance_ratio_
cum = np.cumsum(evr)

print("\n=== PCA on row-demeaned residuals (scaled) ===")
for thr in (0.80, 0.90, 0.95):
    k = int(np.searchsorted(cum, thr) + 1)
    print(f"  PCs for ≥{thr:.0%} variance: {k} (cumulative {cum[k - 1]:.4f})")
print(f"  PC1–3 share: {evr[0]:.4f}, {evr[1]:.4f}, {evr[2]:.4f} (sum first 3: {cum[2]:.4f})")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(np.arange(1, len(cum) + 1), cum)
ax[0].axhline(0.9, ls="--", color="k", alpha=0.5)
ax[0].axhline(0.95, ls=":", color="k", alpha=0.5)
ax[0].set_xlabel("Number of PCs")
ax[0].set_ylabel("Cumulative explained variance")
ax[0].set_title("PCA on row-demeaned residuals")
ax[0].set_ylim(0, 1.02)

m = min(20, len(evr))
ax[1].plot(np.arange(1, m + 1), evr[:m], marker="o", ms=3)
ax[1].set_xlabel("PC")
ax[1].set_ylabel("Explained variance ratio")
ax[1].set_title("Scree (first 20 PCs)")
plt.tight_layout()
plt.show()


#### PC loadings (which columns drive the factors?)

On **scaled** residual data, each PC is a direction in **column space**. **Large absolute loadings** on the **leading** PCs mean that column’s residual series **co-moves** with the main **shared** patterns (cross-sectional factors). **Small** loadings on those PCs mean the column’s residuals are **not** aligned with those directions — more **idiosyncratic** / “formless” in factor space.

This is **heuristic** for Problem 1 (indices vs farmers are unknown): **indices** in the PDF are *blends* of farmers, so we might expect them to sit **on** the shared cone; **nothing** here is a proof. We report:

- **|loading on PC1|** (rank columns).
- **Participation mass:** sum of squared loadings on the **first K PCs**, where **K** is the smaller of (i) the number of PCs needed for **≥75%** cumulative explained variance and (ii) **30** (cap). So you get at least 75% variance in the PC basis when that fits in 30 PCs; otherwise mass uses **30 PCs** and the printout shows actual cumulative variance.

Run after the PCA cell above (or this cell refits the same pipeline from `DF`).


In [ ]:

price_cols = DF.select_dtypes(include=[np.number]).columns
X_raw = DF[price_cols].copy()
row_means = X_raw.mean(axis=1)
R = X_raw.sub(row_means, axis=0).fillna(0.0)

pipe_res = Pipeline(
    [
        ("scale", StandardScaler()),
        ("pca", PCA()),
    ]
)
pipe_res.fit(R)
pca = pipe_res.named_steps["pca"]
names = list(R.columns)

# Participation mass: PCs until 75% cumulative variance, capped at 30 PCs
TARGET_VAR = 0.75
MAX_PCS = 30
cum = np.cumsum(pca.explained_variance_ratio_)
K_75 = int(np.searchsorted(cum, TARGET_VAR) + 1)
K_75 = max(K_75, 1)
K_use = min(K_75, MAX_PCS, pca.n_components_)

# Loadings: variable j on PC k  ~  component[k,j] * sqrt(eigenvalue_k)
comp = pca.components_[:K_use]
ev = pca.explained_variance_[:K_use]
L = comp.T * np.sqrt(ev)
participation_mass = np.sum(L**2, axis=1)
pc1_abs = np.abs(pca.components_[0, :])

load_df = pd.DataFrame(
    {
        "abs_loading_PC1": pc1_abs,
        "sq_loading_mass": participation_mass,
    },
    index=names,
)

print(
    f"Participation mass = sum of squared loadings on first {K_use} PCs. "
    f"Rule: min(PCs for ≥{TARGET_VAR:.0%} var, {MAX_PCS}) → "
    f"K_75={K_75}, K_use={K_use}. "
    f"Cumulative explained variance over those PCs: {cum[K_use - 1]:.4f}."
)
print(load_df.describe())
print("\nTop 20 by |PC1 loading|:")
print(load_df.sort_values("abs_loading_PC1", ascending=False).head(20))
print("\nBottom 20 by |PC1 loading|:")
print(load_df.sort_values("abs_loading_PC1", ascending=True).head(20))
print(f"\nTop 20 by mass in first {K_use} PCs (squared loadings):")
print(load_df.sort_values("sq_loading_mass", ascending=False).head(20))

fig, axes = plt.subplots(2, 2, figsize=(11, 9))

axes[0, 0].hist(load_df["abs_loading_PC1"], bins=22, color="steelblue", edgecolor="white", alpha=0.9)
axes[0, 0].set_xlabel("|Loading on PC1|")
axes[0, 0].set_ylabel("Number of columns")
axes[0, 0].set_title("Residual PCA: strength on PC1")

axes[0, 1].hist(load_df.iloc[:, 1], bins=22, color="darkorange", edgecolor="white", alpha=0.9)
axes[0, 1].set_xlabel(f"Sum of squared loadings (first {K_use} PCs)")
axes[0, 1].set_ylabel("Number of columns")
axes[0, 1].set_title(
    f"Participation mass (≤{MAX_PCS} PCs, aim ≥{TARGET_VAR:.0%} cum. var; actual {cum[K_use - 1]:.1%})"
)

s1 = load_df.sort_values("abs_loading_PC1", ascending=False).reset_index()
axes[1, 0].barh(s1["index"][:20], s1["abs_loading_PC1"][:20], color="steelblue", alpha=0.85)
axes[1, 0].invert_yaxis()
axes[1, 0].set_xlabel("|PC1 loading|")
axes[1, 0].set_title("Top 20 columns on PC1")

col_mass = load_df.columns[1]
s2 = load_df.sort_values(col_mass, ascending=False).reset_index()
axes[1, 1].barh(s2["index"][:20], s2[col_mass][:20], color="darkorange", alpha=0.85)
axes[1, 1].invert_yaxis()
axes[1, 1].set_xlabel("Squared-loading mass")
axes[1, 1].set_title(f"Top 20 by mass in first {K_use} PCs")
plt.tight_layout()
plt.show()

print(load_df.sort_values(col_mass, ascending=False).head(15))


### How to read the residual PCA (exploratory)

In this notebook’s PCA on **row-demeaned residuals**, the output above reports how many PCs hit each variance threshold—often on the order of **~40–45 PCs** to reach **~90%** cumulative variance (use the exact number from your run). That is a statement about **how many orthogonal directions** you need to summarize **co-movement in relative prices** (after stripping out the common cross-sectional level each day)—not about “true” economic objects, but it is still useful intuition.

**Takeaway:** Relative-price “noise” is **high-dimensional**: there are many distinct patterns of who is cheap or rich versus the daily average, not just one or two factors. You can loosely think of each additional PC as capturing another **characterization of cross-sectional deviation** (another way the panel can wiggle away from the row mean).

**Relating to farmers vs indices (speculative):** The challenge mixes **many individual farmer series** with a **smaller set of index-like series**. A PC count in the forties is **broadly consistent** with a market where lots of idiosyncratic series each contribute their own variation, while a handful of indices might load on a subset of shared factors—but **PCs are not labels**: you cannot equate “43 PCs” to “43 farmers” (or “~10–12 indices”) without inspecting **loadings** and domain structure. Treat this as **supporting evidence for rich heterogeneity** in the residuals, not a headcount.


### Distribution of residual variance (per column)

**Residual** on day `t` for column `j`: take that day’s price minus the **average of all observed prices that day** (same day, across the row). That residual is how far that series sits above or below the **cross-sectional average** for that day.

**Residual variance** for a column = variance of those residuals **on days where that price is observed**. Missing days are **excluded** (we do not impute levels or zero out residuals for variance here).

We do **not** label columns as “index” vs “farmer” (that mapping isn’t given).

The plots show the **distribution of these per-column variances** (natural scale and log10 if helpful).


In [ ]:

price_cols = DF.select_dtypes(include=[np.number]).columns
X_raw = DF[price_cols].copy()

# r_ij = price_ij minus cross-sectional mean on day i (mean over non-NaN in that row)
row_means = X_raw.mean(axis=1)
R = X_raw.sub(row_means, axis=0)

# Variance of residuals using only days where that column has a price (ignore missing days)
resid_var = R.apply(
    lambda s: s.dropna().var(ddof=1) if s.notna().sum() > 1 else np.nan
)

print("Per-column residual variance (missing days ignored).")
print(resid_var.describe())

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fig.suptitle(
    "Distribution of residual variance (observed days only)\n"
    "Residual = price minus daily cross-sectional mean; variance over non-missing days per column",
    fontsize=11,
)

v = resid_var.dropna()
axes[0].hist(v, bins=28, color="steelblue", edgecolor="white", alpha=0.9)
axes[0].set_xlabel("Variance of residuals")
axes[0].set_ylabel("Number of columns")
axes[0].set_title("Histogram")

axes[1].hist(np.log10(v + 1e-12), bins=28, color="steelblue", edgecolor="white", alpha=0.9)
axes[1].set_xlabel(r"$\log_{10}(\mathrm{variance} + 10^{-12})$")
axes[1].set_ylabel("Number of columns")
axes[1].set_title("Histogram (log10 scale)")

plt.tight_layout()
plt.show()

residual_variance_by_col = pd.DataFrame({"residual_var_observed_only": resid_var})
# residual_variance_by_col.sort_values("residual_var_observed_only", ascending=False)


## Residual noise analysis

We study **row-mean residuals** $r_{t,j} = p_{t,j} - \bar{p}_t$ where $\bar{p}_t$ is the cross-sectional mean of **observed** prices that day.

The cells below are intentionally small: (1) build `R`, (2) **bias** (average residual per column vs 0), (3) **pooled** marginal distribution, (4) **shape** (skew/kurtosis per column), (5) **spectrum** (periodogram flatness ≈ white noise), (6) **short-lag autocorrelation** (near 0 ≈ no memory).

**Run this section top to bottom** — later code uses `R`, and the last table uses `flat_s` from the spectrum cell.


In [ ]:
# Shared: residual matrix R (NaN where price missing — do not fill here)
price_cols = DF.select_dtypes(include=[np.number]).columns
X_raw = DF[price_cols].copy()
row_means = X_raw.mean(axis=1)
R = X_raw.sub(row_means, axis=0)
R.shape


### 1. Bias: is the average residual above or below zero?

For each column we take the **time average** of $r_{t,j}$ using **non-missing** days only. Values **systematically positive/negative** mean that series tends to sit **above/below** the daily cross-sectional average (after accounting for missingness in the mean).


In [ ]:

col_mean_resid = R.mean(axis=0, skipna=True)
print(col_mean_resid.describe())
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(col_mean_resid.dropna(), bins=20, color="steelblue", edgecolor="white", alpha=0.9)
ax.axvline(0.0, color="k", linestyle="--", linewidth=1)
ax.set_xlabel("Mean residual (time average per column)")
ax.set_ylabel("Number of columns")
ax.set_title("Distribution of column-mean residuals (observed days only)")
plt.tight_layout()
plt.show()
print(col_mean_resid.sort_values().head(15))


### 2. Pooled residuals: overall shape

All **observed** residual values (every column, every day) pooled into one sample. This is **not** i.i.d. (rows share the same day), but it shows whether mass is roughly centered and symmetric vs heavy-tailed.


In [ ]:
pooled = R.stack().dropna()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(pooled, bins=80, density=True, color="gray", edgecolor="white", alpha=0.85)
axes[0].axvline(0.0, color="k", linestyle="--", linewidth=1)
axes[0].set_xlabel("Residual")
axes[0].set_ylabel("Density")
axes[0].set_title("Pooled residuals (all columns, observed)")

# Zoom center (exclude extreme tails for readability)
q = pooled.quantile([0.01, 0.99])
m = pooled.between(q.iloc[0], q.iloc[1])
axes[1].hist(pooled[m], bins=60, density=True, color="gray", edgecolor="white", alpha=0.85)
axes[1].axvline(0.0, color="k", linestyle="--", linewidth=1)
axes[1].set_xlabel("Residual")
axes[1].set_title("Same, zoomed to 1%–99% range")
plt.tight_layout()
plt.show()

print("Pooled mean:", pooled.mean(), "std:", pooled.std())


### 3. Per-column shape: skewness and excess kurtosis

**Skew** $\neq 0$: asymmetry. **Excess kurtosis** $> 0$: heavier tails than Gaussian. Computed per column on **observed** residuals (pandas `skipna`).


In [ ]:
sk = R.skew(axis=0, skipna=True)
ku = R.kurtosis(axis=0, skipna=True)  # excess kurtosis (normal = 0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(sk.dropna(), bins=20, color="coral", edgecolor="white", alpha=0.9)
axes[0].axvline(0.0, color="k", linestyle="--", linewidth=1)
axes[0].set_xlabel("Skewness")
axes[0].set_title("Per-column skewness of residuals")

axes[1].hist(ku.dropna(), bins=20, color="seagreen", edgecolor="white", alpha=0.9)
axes[1].axvline(0.0, color="k", linestyle="--", linewidth=1)
axes[1].set_xlabel("Excess kurtosis")
axes[1].set_title("Per-column excess kurtosis")
plt.tight_layout()
plt.show()


### 4. Spectrum: periodogram “flatness” (white noise → flat PSD)

For **Gaussian white noise**, the **power spectral density** is **flat** across frequencies. A common scalar summary is **spectral flatness** (Wiener):

$$
\frac{\exp\bigl(\mathbb{E}[\log S(f)]\bigr)}{\mathbb{E}[S(f)]}
$$

in $(0,1]$, with values **closer to 1** when the spectrum is **more uniform** (more “noise-like”). We compute the **periodogram** per column on **observed-only** residual series (constant detrend), then flatness. Short series give noisy estimates — treat as exploratory.


In [ ]:


def periodogram_flatness(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size < 64:
        return np.nan
    _, pxx = periodogram(x, detrend="constant")
    pxx = np.maximum(pxx, 1e-20)
    return float(np.exp(np.mean(np.log(pxx))) / np.mean(pxx))


flatness = {}
for col in R.columns:
    flatness[col] = periodogram_flatness(R[col].dropna().to_numpy())

flat_s = pd.Series(flatness, name="spectral_flatness")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(flat_s.dropna(), bins=18, color="mediumpurple", edgecolor="white", alpha=0.9)
ax.axvline(1.0, color="k", linestyle=":", linewidth=1, label="perfect flatness (idealized)")
ax.set_xlabel("Spectral flatness")
ax.set_ylabel("Number of columns")
ax.set_title("Periodogram flatness per column (higher ≈ more white-noise-like PSD)")
ax.legend()
plt.tight_layout()
plt.show()

print(flat_s.describe())
print("\nColumns closest to flat spectrum (top 10):")
print(flat_s.sort_values(ascending=False).head(10))


### 5. Lag-1 autocorrelation (linear memory)

Under **i.i.d.** noise, **lag-1 correlation** in time should be **near zero**. We correlate **r_t** with **r_{t-1}** using only rows where **both** are observed (calendar-aligned), so missing days do not shift the lag.


In [ ]:

acf1 = pd.Series({col: lag1_acf_calendar(R[col]) for col in R.columns}, name="lag1_acf")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(acf1.dropna(), bins=22, color="teal", edgecolor="white", alpha=0.9)
ax.axvline(0.0, color="k", linestyle="--", linewidth=1)
ax.set_xlabel("Lag-1 autocorrelation (calendar-aligned)")
ax.set_ylabel("Number of columns")
ax.set_title("Residuals: linear persistence at lag 1")
plt.tight_layout()
plt.show()

print(acf1.describe())
print(
    pd.DataFrame({"spectral_flatness": flat_s, "lag1_acf": acf1})
    .sort_values("spectral_flatness", ascending=False)
    .head(12)
    .to_string()
)


**Three code cells below:** (1) row-mean residuals, (2) **level prices** (imputed), (3) first differences and **comparison** across all three.

# Non-negative least squares (leave-one-out $R^2$)

**From the challenge PDF:** each *index* price is a **non-negative linear combination** of **farmer** flour prices (coefficients $c_{ij} \ge 0$); column labels are unknown. That motivates asking: which columns look like **blendable** from the others with **non-negative** weights?

**Setup:** For each column $j$, treat it as the response and **all other columns** as features (**no intercept**). Solve $\min_{\beta \ge 0} \| y - X\beta \|_2^2$ with **`scipy.optimize.nnls`**, then report

$$
R^2 = 1 - \frac{\sum_i (y_i - \hat y_i)^2}{\sum_i (y_i - \bar y)^2}
$$

using the **same** rows (training fit). This is **exploratory**.

### Why can $R^2$ be negative?

Here $R^2$ is **not** the usual OLS $R^2$ with an intercept. NNLS forces **$\beta \ge 0$** and **no intercept**, so the fitted $\hat y$ can be a **worse** predictor than the **constant mean** $\bar y$. When $\mathrm{SS}_\mathrm{res} > \mathrm{SS}_\mathrm{tot}$, the formula above gives **$R^2 < 0$**. That means “constrained cone fit is worse than predicting the column mean.” It is **not** a bug.

**Three panels:**

1. **Row-mean residuals** — subtract cross-sectional daily mean; **missing → 0** (complete matrix).
2. **Level prices** — **row-mean imputation** for missing cells (same as elsewhere), then NNLS on **levels** (not differences).
3. **First differences** — $\Delta p_t$; **NaN → 0** for missing diffs.

Run cells **in order** for the combined comparison table and scatter plots.


In [ ]:

price_cols = DF.select_dtypes(include=[np.number]).columns
X_raw = DF[price_cols].copy()

# Row-mean residuals; missing -> 0 (same as PCA on R)
row_means = X_raw.mean(axis=1)
R = X_raw.sub(row_means, axis=0).fillna(0.0)
Z_resid = R.to_numpy(dtype=np.float64)
names = list(price_cols)




r2_resid = nnls_leave_one_out_r2(Z_resid, names)

print("NNLS on row-mean residuals — leave-one-out R^2")
print("(higher = easier to match others; R^2 can be negative — see markdown.)\n")
print(r2_resid.describe())
print("\nTop 15 columns by R^2:")
print(r2_resid.sort_values(ascending=False).head(15))
print("\nBottom 15 columns by R^2:")
print(r2_resid.sort_values(ascending=True).head(15))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].hist(r2_resid.dropna(), bins=22, color="steelblue", edgecolor="white", alpha=0.9)
axes[0].set_xlabel(r"$R^2$")
axes[0].set_ylabel("Number of columns")
axes[0].set_title("NNLS R^2 — row-mean residuals")

sorted_r = r2_resid.sort_values(ascending=False).rename_axis("column").reset_index()
axes[1].barh(sorted_r["column"][:20], sorted_r["r2_nnls"][:20], color="steelblue", alpha=0.85)
axes[1].set_xlabel(r"$R^2$")
axes[1].set_title("Top 20 columns")
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

r2_resid


In [ ]:

price_cols = DF.select_dtypes(include=[np.number]).columns
X_raw = DF[price_cols].copy()
names = list(price_cols)

# Level prices: fill missing with row (day) mean so NNLS has a full matrix
row_means = X_raw.mean(axis=1)
X_level = X_raw.T.fillna(row_means).T.fillna(0.0)
Z_level = X_level.to_numpy(dtype=np.float64)




r2_level = nnls_leave_one_out_r2(Z_level, names)

print("NNLS on level prices (row-mean imputed missing) — leave-one-out R^2")
print("(R^2 can be negative — see markdown above.)\n")
print(r2_level.describe())
print("\nTop 15 columns by R^2:")
print(r2_level.sort_values(ascending=False).head(15))
print("\nBottom 15 columns by R^2:")
print(r2_level.sort_values(ascending=True).head(15))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].hist(r2_level.dropna(), bins=22, color="goldenrod", edgecolor="white", alpha=0.9)
axes[0].set_xlabel(r"$R^2$")
axes[0].set_ylabel("Number of columns")
axes[0].axvline(0.0, color="k", linestyle="--", linewidth=1)
axes[0].set_title("NNLS R^2 — level prices")

sorted_l = r2_level.sort_values(ascending=False).rename_axis("column").reset_index()
axes[1].barh(sorted_l["column"][:20], sorted_l["r2_nnls"][:20], color="goldenrod", alpha=0.85)
axes[1].set_xlabel(r"$R^2$")
axes[1].set_title("Top 20 columns")
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

r2_level


In [ ]:

# Run the **residuals** and **levels** NNLS cells above first for the full comparison.
price_cols = DF.select_dtypes(include=[np.number]).columns
X_raw = DF[price_cols].copy()
names = list(price_cols)

D = X_raw.diff().fillna(0.0)
Z_diff = D.to_numpy(dtype=np.float64)




r2_diff = nnls_leave_one_out_r2(Z_diff, names)

print("NNLS on first differences — leave-one-out R^2\n")
print(r2_diff.describe())
print("\nTop 15 columns by R^2:")
print(r2_diff.sort_values(ascending=False).head(15))
print("\nBottom 15 columns by R^2:")
print(r2_diff.sort_values(ascending=True).head(15))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].hist(r2_diff.dropna(), bins=22, color="darkseagreen", edgecolor="white", alpha=0.9)
axes[0].axvline(0.0, color="k", linestyle="--", linewidth=1)
axes[0].set_xlabel(r"$R^2$")
axes[0].set_ylabel("Number of columns")
axes[0].set_title("NNLS R^2 — first differences")

sorted_d = r2_diff.sort_values(ascending=False).rename_axis("column").reset_index()
axes[1].barh(sorted_d["column"][:20], sorted_d["r2_nnls"][:20], color="darkseagreen", alpha=0.85)
axes[1].set_xlabel(r"$R^2$")
axes[1].set_title("Top 20 columns")
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

have_r = "r2_resid" in globals()
have_l = "r2_level" in globals()
if not have_r or not have_l:
    print("Run the residuals NNLS cell and the level-prices NNLS cell above for the 3-way summary and scatters.")
if have_r and have_l:
    summary = pd.DataFrame(
        {
            "r2_residuals_rowmean": r2_resid,
            "r2_level_prices": r2_level,
            "r2_first_diff": r2_diff,
        }
    )
    print("\nCombined summary (describe):")
    print(summary.describe())

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    lo = summary.min().min()
    hi = summary.max().max()
    pad = 0.05 * (hi - lo) if hi > lo else 0.05

    axes[0].scatter(summary["r2_residuals_rowmean"], summary["r2_level_prices"], alpha=0.75, edgecolors="none")
    axes[0].plot([lo - pad, hi + pad], [lo - pad, hi + pad], "k--", lw=0.8, alpha=0.5)
    axes[0].set_xlabel("Residuals R^2")
    axes[0].set_ylabel("Levels R^2")
    axes[0].set_title("Residual vs level")

    axes[1].scatter(summary["r2_residuals_rowmean"], summary["r2_first_diff"], alpha=0.75, edgecolors="none")
    axes[1].plot([lo - pad, hi + pad], [lo - pad, hi + pad], "k--", lw=0.8, alpha=0.5)
    axes[1].set_xlabel("Residuals R^2")
    axes[1].set_ylabel("First-diff R^2")
    axes[1].set_title("Residual vs first-diff")

    axes[2].scatter(summary["r2_level_prices"], summary["r2_first_diff"], alpha=0.75, edgecolors="none")
    axes[2].plot([lo - pad, hi + pad], [lo - pad, hi + pad], "k--", lw=0.8, alpha=0.5)
    axes[2].set_xlabel("Levels R^2")
    axes[2].set_ylabel("First-diff R^2")
    axes[2].set_title("Level vs first-diff")

    plt.tight_layout()
    plt.show()

    print(summary.sort_values("r2_residuals_rowmean", ascending=False).head(15))
